# Optimization examples for liquid-phase separations

**HPLC 2026 — Introduction to Artificial Intelligence for Liquid-phase Separations**

**Prepared for the course by Dr. Tijmen S. Bos.**  
This notebook is part of the course **Introduction to Artificial Intelligence for Liquid-phase Separations** and is intended as browser-based teaching material for the HPLC 2026 course.

**Company affiliation.**  
This material is part of **Innovative Data Evaluation And Separations (IDEAS)**, Dr. Tijmen S. Bos's company for data evaluation and separations education/consultancy. Website: **bos-ideas.com**.

**Ownership and use.**  
The educational content in this notebook belongs to **Dr. Tijmen S. Bos / Innovative Data Evaluation And Separations (IDEAS)**, unless a different source is explicitly indicated. The examples are provided for course participation, demonstration, and educational discussion.

**Non-liability and educational-use note.**  
This notebook is provided for course teaching, demonstration, and educational discussion only. The examples use simplified and/or synthetic data and are not validated analytical methods. They should not be used as the sole basis for experimental, regulatory, quality-control, safety, commercial, medical, legal, or financial decisions. Dr. Tijmen S. Bos and Innovative Data Evaluation And Separations (IDEAS) accept no liability for decisions, losses, or damages arising from use, modification, or interpretation of this material. Users remain responsible for validating any method, model, or workflow before practical application.

This browser-friendly JupyterLite exercise covers optimization concepts from the course slides:

- **Grid and random search** as simple baselines.
- **Gradient ascent** as a local optimizer.
- **Particle swarm optimization** as a multi-start/population strategy.
- **Bayesian optimization** using a Gaussian-process surrogate and acquisition rule.
- **Multi-fidelity optimization** using many cheap simulations and fewer expensive experiments.

The examples use synthetic LC-MS method-development response surfaces. They are teaching examples, not validated chromatographic models.


## 1. Imports, palette, and synthetic optimization problem

The notebook is JupyterLite-safe: no external data files, no `ipywidgets`, and no deep-learning libraries. The interactive exercise uses browser-side JavaScript after the data payload has been generated by Python.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display

RANDOM_STATE = 7

# IDEAS/HPLC 2026 palette
IDEAS_CYAN = "#43CEFF"
IDEAS_MAGENTA = "#DB3FC8"
IDEAS_GOLD = "#FFC000"
IDEAS_GREEN = "#92D050"
IDEAS_TEXT = "#16495A"
IDEAS_GRID = "#E5F8FD"
IDEAS_PALE_CYAN = "#D9F7FF"
IDEAS_PALE_MAGENTA = "#F8D8F4"
IDEAS_PALE_GOLD = "#FFF2BF"
IDEAS_PALE_GREEN = "#E7F5D9"

OPT_CMAP = LinearSegmentedColormap.from_list(
    "ideas_optimization",
    ["#FFFFFF", IDEAS_PALE_CYAN, IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GOLD],
)

plt.rcParams.update({
    "axes.edgecolor": IDEAS_TEXT,
    "axes.labelcolor": IDEAS_TEXT,
    "axes.titlecolor": IDEAS_TEXT,
    "xtick.color": IDEAS_TEXT,
    "ytick.color": IDEAS_TEXT,
    "grid.color": IDEAS_GRID,
    "grid.alpha": 0.75,
    "legend.frameon": True,
})

X_LABEL = "Gradient time (min)"
Y_LABEL = "Final organic modifier (%B)"
X_RANGE = (5.0, 60.0)
Y_RANGE = (40.0, 95.0)


## 2. Synthetic chromatographic response function

In method development, the optimizer needs a numerical objective. The slides frame this as the problem of mathematically capturing how good a separation is. Here, the objective is a synthetic **chromatographic response function** (CRF), where high values mean better overall method quality.

Two surfaces are used:

- **Smooth response surface** — easier for local gradient-based optimization.
- **Rugged CRF surface** — contains several local optima, illustrating why global or model-based strategies are often needed.


In [ ]:
def to_method_units(uv):
    uv = np.asarray(uv)
    x = X_RANGE[0] + uv[..., 0] * (X_RANGE[1] - X_RANGE[0])
    y = Y_RANGE[0] + uv[..., 1] * (Y_RANGE[1] - Y_RANGE[0])
    return np.stack([x, y], axis=-1)


def gaussian_2d(u, v, u0, v0, su, sv, amp=1.0):
    return amp * np.exp(-0.5 * (((u - u0) / su) ** 2 + ((v - v0) / sv) ** 2))


def crf_surface_uv(uv, complexity="rugged", fidelity="high"):
    """Synthetic chromatographic response function. Higher is better."""
    uv = np.asarray(uv)
    u = uv[..., 0]
    v = uv[..., 1]

    smooth = (
        gaussian_2d(u, v, 0.62, 0.44, 0.18, 0.16, 1.05)
        + gaussian_2d(u, v, 0.34, 0.72, 0.22, 0.14, 0.38)
        + 0.10 * np.exp(-2.0 * u)
    )
    penalty = 0.16 * u + 0.06 * (v - 0.55) ** 2 + 0.04 * np.maximum(0, 0.18 - u)
    score = smooth - penalty

    if complexity == "rugged":
        score += (
            gaussian_2d(u, v, 0.22, 0.78, 0.10, 0.12, 0.52)
            + gaussian_2d(u, v, 0.74, 0.24, 0.09, 0.10, 0.46)
            + gaussian_2d(u, v, 0.48, 0.60, 0.06, 0.06, 0.26)
            - gaussian_2d(u, v, 0.52, 0.48, 0.07, 0.08, 0.28)
            + 0.08 * np.sin(5.0 * np.pi * u) * np.cos(4.0 * np.pi * v)
            + 0.05 * np.sin(9.0 * np.pi * (u + 0.2 * v))
        )

    if fidelity == "low":
        # Low fidelity is correlated with high fidelity, but biased and smoother.
        score = (
            0.88 * score
            + 0.10 * gaussian_2d(u, v, 0.56, 0.50, 0.24, 0.22, 1.0)
            - 0.05 * gaussian_2d(u, v, 0.72, 0.28, 0.15, 0.15, 1.0)
            + 0.025 * np.sin(2.0 * np.pi * u)
        )
    return score


def normalized_score(uv, complexity="rugged", fidelity="high"):
    raw = crf_surface_uv(uv, complexity=complexity, fidelity=fidelity)
    return np.clip((raw + 0.12) / 1.32, 0, 1)


def make_surface_grid(complexity="rugged", fidelity="high", n=90):
    u = np.linspace(0, 1, n)
    v = np.linspace(0, 1, n)
    uu, vv = np.meshgrid(u, v)
    uv = np.stack([uu.ravel(), vv.ravel()], axis=1)
    z = normalized_score(uv, complexity=complexity, fidelity=fidelity).reshape(n, n)
    xy = to_method_units(uv).reshape(n, n, 2)
    return xy[:, :, 0], xy[:, :, 1], z


def best_grid_point(complexity="rugged", fidelity="high", n=180):
    u = np.linspace(0, 1, n)
    v = np.linspace(0, 1, n)
    uu, vv = np.meshgrid(u, v)
    uv = np.stack([uu.ravel(), vv.ravel()], axis=1)
    scores = normalized_score(uv, complexity=complexity, fidelity=fidelity)
    idx = int(np.argmax(scores))
    return uv[idx], float(scores[idx])

print("Synthetic CRF functions loaded.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
for ax, complexity in zip(axes, ["smooth", "rugged"]):
    xx, yy, zz = make_surface_grid(complexity=complexity, n=100)
    im = ax.contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)
    best_uv, best_score = best_grid_point(complexity=complexity)
    best_xy = to_method_units(best_uv)
    ax.scatter(best_xy[0], best_xy[1], s=85, marker="*", color=IDEAS_GOLD, edgecolor=IDEAS_TEXT, label="grid optimum")
    ax.set_xlabel(X_LABEL)
    ax.set_ylabel(Y_LABEL)
    ax.set_title(f"{complexity.capitalize()} synthetic CRF surface")
    ax.grid(True, linewidth=0.6)
    ax.legend(loc="lower right", fontsize=8)

# Use a dedicated colorbar axis outside the two panels.
# This avoids the JupyterLite/Matplotlib issue where a shared colorbar can
# be squeezed into the middle of the subplot area.
fig.subplots_adjust(left=0.07, right=0.87, bottom=0.15, top=0.90, wspace=0.26)
cbar_ax = fig.add_axes([0.895, 0.18, 0.018, 0.66])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Normalized CRF score")
plt.show()


## 3. Baseline searches: grid and random search

Grid search is systematic but scales poorly with dimensionality. Random search is simple and often surprisingly effective, but it does not learn from previous observations.


In [ ]:
def grid_search_demo(complexity="rugged", n_per_axis=14):
    u = np.linspace(0, 1, n_per_axis)
    v = np.linspace(0, 1, n_per_axis)
    uu, vv = np.meshgrid(u, v)
    uv = np.stack([uu.ravel(), vv.ravel()], axis=1)
    scores = normalized_score(uv, complexity=complexity)
    idx = int(np.argmax(scores))
    return uv, scores, idx


def random_search_demo(complexity="rugged", n_samples=80, random_state=7):
    rng = np.random.default_rng(random_state)
    uv = rng.uniform(0, 1, size=(n_samples, 2))
    scores = normalized_score(uv, complexity=complexity)
    idx = int(np.argmax(scores))
    return uv, scores, idx

complexity = "rugged"
grid_uv, grid_scores, grid_idx = grid_search_demo(complexity=complexity, n_per_axis=12)
rand_uv, rand_scores, rand_idx = random_search_demo(complexity=complexity, n_samples=80)
xx, yy, zz = make_surface_grid(complexity=complexity, n=100)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
for ax, uv, scores, idx, title, color in [
    (axes[0], grid_uv, grid_scores, grid_idx, "Grid search", IDEAS_CYAN),
    (axes[1], rand_uv, rand_scores, rand_idx, "Random search", IDEAS_MAGENTA),
]:
    ax.contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)
    xy = to_method_units(uv)
    ax.scatter(xy[:, 0], xy[:, 1], s=20, color=color, edgecolor="white", linewidth=0.3, alpha=0.82)
    best_xy = to_method_units(uv[idx])
    ax.scatter(best_xy[0], best_xy[1], s=95, marker="*", color=IDEAS_GOLD, edgecolor=IDEAS_TEXT)
    ax.set_xlabel(X_LABEL)
    ax.set_ylabel(Y_LABEL)
    ax.set_title(f"{title}: best score = {scores[idx]:.3f}")
    ax.grid(True, linewidth=0.6)

plt.tight_layout()
plt.show()

pd.DataFrame({
    "Method": ["Grid search", "Random search"],
    "Evaluations": [len(grid_scores), len(rand_scores)],
    "Best normalized CRF score": [grid_scores[grid_idx], rand_scores[rand_idx]],
    "Best gradient time (min)": [to_method_units(grid_uv[grid_idx])[0], to_method_units(rand_uv[rand_idx])[0]],
    "Best final %B": [to_method_units(grid_uv[grid_idx])[1], to_method_units(rand_uv[rand_idx])[1]],
}).round(3)


## 4. Gradient ascent: efficient but local

The slide exercise asks what happens when a gradient-descent algorithm starts at a point on a rugged landscape. Here we use **gradient ascent** because we maximize the CRF. The principle is the same: the algorithm follows local slope information and can become trapped in a local optimum.


In [ ]:
def finite_difference_gradient(uv, complexity="rugged", eps=1e-3):
    uv = np.asarray(uv, dtype=float)
    grad = np.zeros(2)
    for j in range(2):
        step = np.zeros(2)
        step[j] = eps
        plus = np.clip(uv + step, 0, 1)
        minus = np.clip(uv - step, 0, 1)
        grad[j] = (normalized_score(plus, complexity=complexity) - normalized_score(minus, complexity=complexity)) / (2 * eps)
    return grad


def gradient_ascent(complexity="rugged", start=(0.12, 0.18), step_size=0.055, n_steps=34):
    uv = np.asarray(start, dtype=float)
    path = [uv.copy()]
    for _ in range(n_steps):
        grad = finite_difference_gradient(uv, complexity=complexity)
        norm = np.linalg.norm(grad)
        if norm < 1e-8:
            break
        uv = np.clip(uv + step_size * grad / (norm + 1e-12), 0, 1)
        path.append(uv.copy())
    path = np.asarray(path)
    scores = normalized_score(path, complexity=complexity)
    return path, scores

starts = [(0.12, 0.18), (0.85, 0.18), (0.22, 0.86)]
xx, yy, zz = make_surface_grid(complexity="rugged", n=100)
plt.figure(figsize=(7.2, 5.7))
plt.contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)

for start, color in zip(starts, [IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GREEN]):
    path, scores = gradient_ascent(complexity="rugged", start=start, step_size=0.055, n_steps=30)
    xy = to_method_units(path)
    plt.plot(xy[:, 0], xy[:, 1], "-o", color=color, markersize=3.5, linewidth=1.6, label=f"start score {scores[0]:.2f} → {scores[-1]:.2f}")
    plt.scatter(xy[0, 0], xy[0, 1], s=70, marker="X", color=color, edgecolor="white", linewidth=0.8)

best_uv, _ = best_grid_point(complexity="rugged")
best_xy = to_method_units(best_uv)
plt.scatter(best_xy[0], best_xy[1], s=120, marker="*", color=IDEAS_GOLD, edgecolor=IDEAS_TEXT, label="global grid optimum")
plt.xlabel(X_LABEL)
plt.ylabel(Y_LABEL)
plt.title("Gradient ascent follows local slope information")
plt.grid(True, linewidth=0.6)
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()


**Interpretation.**  
Gradient-based optimization can be highly efficient on a smooth surface, but it is local. On a rugged CRF surface, the final answer depends strongly on the starting point.


## 5. Particle swarm optimization

Particle swarm optimization is a population-based search method. Each particle keeps track of its own best position while also moving toward the best position found by the swarm.


In [ ]:
def particle_swarm(complexity="rugged", n_particles=18, n_iter=32, inertia=0.66, cognitive=1.2, social=1.35, random_state=7):
    rng = np.random.default_rng(random_state)
    pos = rng.uniform(0, 1, size=(n_particles, 2))
    vel = rng.normal(0, 0.08, size=(n_particles, 2))
    scores = normalized_score(pos, complexity=complexity)
    personal_best = pos.copy()
    personal_score = scores.copy()
    g_idx = int(np.argmax(scores))
    global_best = pos[g_idx].copy()
    global_score = float(scores[g_idx])
    history = [pos.copy()]
    best_history = [global_best.copy()]
    score_history = [global_score]

    for _ in range(n_iter):
        r1 = rng.uniform(0, 1, size=(n_particles, 2))
        r2 = rng.uniform(0, 1, size=(n_particles, 2))
        vel = inertia * vel + cognitive * r1 * (personal_best - pos) + social * r2 * (global_best - pos)
        vel = np.clip(vel, -0.16, 0.16)
        pos = np.clip(pos + vel, 0, 1)
        scores = normalized_score(pos, complexity=complexity)
        improved = scores > personal_score
        personal_best[improved] = pos[improved]
        personal_score[improved] = scores[improved]
        g_idx = int(np.argmax(personal_score))
        if personal_score[g_idx] > global_score:
            global_best = personal_best[g_idx].copy()
            global_score = float(personal_score[g_idx])
        history.append(pos.copy())
        best_history.append(global_best.copy())
        score_history.append(global_score)
    return np.asarray(history), np.asarray(best_history), np.asarray(score_history)

history, best_history, score_history = particle_swarm(complexity="rugged")
xx, yy, zz = make_surface_grid(complexity="rugged", n=100)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

axes[0].contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)
for t in [0, 5, 12, 24, 32]:
    xy = to_method_units(history[t])
    axes[0].scatter(xy[:, 0], xy[:, 1], s=20 + t, alpha=0.65, label=f"iter {t}")
best_xy = to_method_units(best_history)
axes[0].plot(best_xy[:, 0], best_xy[:, 1], color=IDEAS_TEXT, linewidth=1.8, marker="o", markersize=3, label="best-so-far path")
axes[0].set_xlabel(X_LABEL)
axes[0].set_ylabel(Y_LABEL)
axes[0].set_title("Particle swarm explores with multiple candidate methods")
axes[0].grid(True, linewidth=0.6)
axes[0].legend(fontsize=8, loc="lower right")

axes[1].plot(score_history, color=IDEAS_MAGENTA, linewidth=2.4)
axes[1].scatter(np.arange(len(score_history)), score_history, color=IDEAS_MAGENTA, s=18)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Best normalized CRF score")
axes[1].set_title("Best-so-far score")
axes[1].grid(True, linewidth=0.6)
plt.tight_layout()
plt.show()


## 6. Bayesian optimization with a Gaussian-process surrogate

Bayesian optimization uses a surrogate model to decide where to test next. The loop is: sample the surface, update the model, compute an acquisition function, perform the next experiment, and repeat.

This example uses a NumPy-only Gaussian-process surrogate and an upper-confidence-bound acquisition function:

\[
\mathrm{UCB}(x) = \mu(x) + eta\sigma(x)
\]

where \(\mu\) is the surrogate mean, \(\sigma\) is uncertainty, and \(eta\) controls exploration.


In [ ]:
def rbf_kernel(a, b, length_scale=0.20, variance=1.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a[:, None, :] - b[None, :, :]
    d2 = np.sum(diff**2, axis=2)
    return variance * np.exp(-0.5 * d2 / (length_scale**2))


def gp_posterior(x_train, y_train, x_query, length_scale=0.20, noise=1e-5):
    x_train = np.asarray(x_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    x_query = np.asarray(x_query, dtype=float)
    K = rbf_kernel(x_train, x_train, length_scale=length_scale) + noise * np.eye(len(x_train))
    Ks = rbf_kernel(x_train, x_query, length_scale=length_scale)
    Kss_diag = np.diag(rbf_kernel(x_query, x_query, length_scale=length_scale))
    y_mean = y_train.mean()
    alpha = np.linalg.solve(K, y_train - y_mean)
    mu = y_mean + Ks.T @ alpha
    v = np.linalg.solve(K, Ks)
    var = np.maximum(Kss_diag - np.sum(Ks * v, axis=0), 1e-10)
    return mu, np.sqrt(var)


def bayesian_optimization(complexity="rugged", n_init=5, n_iter=16, beta=2.0, length_scale=0.18, random_state=7, candidate_grid=32):
    rng = np.random.default_rng(random_state)
    x_train = rng.uniform(0.05, 0.95, size=(n_init, 2))
    y_train = normalized_score(x_train, complexity=complexity)
    grid_axis = np.linspace(0.02, 0.98, candidate_grid)
    uu, vv = np.meshgrid(grid_axis, grid_axis)
    candidates = np.stack([uu.ravel(), vv.ravel()], axis=1)
    best_history = [float(y_train.max())]

    for _ in range(n_iter):
        mu, sigma = gp_posterior(x_train, y_train, candidates, length_scale=length_scale)
        acq = mu + beta * sigma
        distances = np.sqrt(((candidates[:, None, :] - x_train[None, :, :]) ** 2).sum(axis=2))
        acq[distances.min(axis=1) < 0.035] = -np.inf
        next_x = candidates[int(np.argmax(acq))]
        next_y = normalized_score(next_x, complexity=complexity)
        x_train = np.vstack([x_train, next_x])
        y_train = np.append(y_train, next_y)
        best_history.append(float(y_train.max()))
    return x_train, y_train, np.asarray(best_history), candidates

x_bo, y_bo, bo_best_history, candidates = bayesian_optimization(complexity="rugged", beta=2.0, n_iter=14)
mu, sigma = gp_posterior(x_bo, y_bo, candidates, length_scale=0.18)
xx, yy, zz = make_surface_grid(complexity="rugged", n=100)
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))

axes[0].contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)
xy = to_method_units(x_bo)
axes[0].plot(xy[:, 0], xy[:, 1], "-o", color=IDEAS_TEXT, markersize=4, linewidth=1.4)
axes[0].scatter(xy[:5, 0], xy[:5, 1], s=55, color=IDEAS_CYAN, edgecolor="white", label="initial")
axes[0].scatter(xy[5:, 0], xy[5:, 1], s=55, color=IDEAS_MAGENTA, edgecolor="white", label="BO-selected")
axes[0].set_title("Bayesian optimization samples")
axes[0].set_xlabel(X_LABEL)
axes[0].set_ylabel(Y_LABEL)
axes[0].grid(True, linewidth=0.6)
axes[0].legend(fontsize=8)

candidate_xy = to_method_units(candidates)
axes[1].tricontourf(candidate_xy[:, 0], candidate_xy[:, 1], mu, levels=22, cmap=OPT_CMAP)
axes[1].scatter(xy[:, 0], xy[:, 1], s=24, color=IDEAS_TEXT, edgecolor="white", linewidth=0.4)
axes[1].set_title("GP posterior mean")
axes[1].set_xlabel(X_LABEL)
axes[1].set_ylabel(Y_LABEL)
axes[1].grid(True, linewidth=0.6)

axes[2].tricontourf(candidate_xy[:, 0], candidate_xy[:, 1], sigma, levels=22, cmap=OPT_CMAP)
axes[2].scatter(xy[:, 0], xy[:, 1], s=24, color=IDEAS_TEXT, edgecolor="white", linewidth=0.4)
axes[2].set_title("GP posterior uncertainty")
axes[2].set_xlabel(X_LABEL)
axes[2].set_ylabel(Y_LABEL)
axes[2].grid(True, linewidth=0.6)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6.5, 4.2))
plt.plot(bo_best_history, color=IDEAS_MAGENTA, linewidth=2.4)
plt.scatter(np.arange(len(bo_best_history)), bo_best_history, color=IDEAS_MAGENTA, s=24)
plt.xlabel("BO iteration")
plt.ylabel("Best normalized CRF score")
plt.title("Bayesian optimization: best-so-far score")
plt.grid(True, linewidth=0.6)
plt.tight_layout()
plt.show()


**Interpretation.**  
Bayesian optimization is useful when each real experiment is expensive. Instead of sampling blindly, it builds a model of the response surface and chooses new experiments based on both predicted quality and uncertainty.


## 7. Multi-fidelity optimization

Multi-fidelity optimization combines many cheap simulations with a smaller number of expensive high-fidelity experiments. The low-fidelity simulator is useful only if it is sufficiently correlated with the true experimental objective.


In [ ]:
def multi_fidelity_search(complexity="rugged", n_sim=220, n_exp=18, random_state=11):
    rng = np.random.default_rng(random_state)
    candidates = rng.uniform(0, 1, size=(n_sim, 2))
    low_scores = normalized_score(candidates, complexity=complexity, fidelity="low")
    ranking_score = low_scores + rng.normal(0, 0.015, size=n_sim)
    shortlist_idx = np.argsort(ranking_score)[-n_exp:]
    high_points = candidates[shortlist_idx]
    high_scores = normalized_score(high_points, complexity=complexity, fidelity="high")
    best_idx = int(np.argmax(high_scores))
    return candidates, low_scores, high_points, high_scores, best_idx

candidates, low_scores, high_points, high_scores, best_idx = multi_fidelity_search()
xx_high, yy_high, zz_high = make_surface_grid(complexity="rugged", fidelity="high", n=100)
xx_low, yy_low, zz_low = make_surface_grid(complexity="rugged", fidelity="low", n=100)
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.6))

for ax, xx, yy, zz, title in [
    (axes[0], xx_low, yy_low, zz_low, "Low-fidelity simulator"),
    (axes[1], xx_high, yy_high, zz_high, "High-fidelity objective"),
]:
    ax.contourf(xx, yy, zz, levels=24, cmap=OPT_CMAP)
    ax.set_xlabel(X_LABEL)
    ax.set_ylabel(Y_LABEL)
    ax.set_title(title)
    ax.grid(True, linewidth=0.6)

cand_xy = to_method_units(candidates)
exp_xy = to_method_units(high_points)
axes[0].scatter(cand_xy[:, 0], cand_xy[:, 1], s=12, color=IDEAS_CYAN, alpha=0.45, label="cheap simulations")
axes[0].scatter(exp_xy[:, 0], exp_xy[:, 1], s=55, color=IDEAS_MAGENTA, edgecolor="white", label="selected for experiments")
axes[0].legend(fontsize=8)
axes[1].scatter(exp_xy[:, 0], exp_xy[:, 1], s=55, color=IDEAS_MAGENTA, edgecolor="white", label="high-fidelity experiments")
best_xy = to_method_units(high_points[best_idx])
axes[1].scatter(best_xy[0], best_xy[1], s=130, marker="*", color=IDEAS_GOLD, edgecolor=IDEAS_TEXT, label="best experiment")
axes[1].legend(fontsize=8)
axes[2].scatter(low_scores[np.argsort(low_scores)[-len(high_scores):]], np.sort(high_scores), color=IDEAS_GREEN, s=45, edgecolor="white")
axes[2].set_xlabel("Low-fidelity score of shortlisted candidates")
axes[2].set_ylabel("High-fidelity score")
axes[2].set_title("Low and high fidelity are correlated, not identical")
axes[2].grid(True, linewidth=0.6)
plt.tight_layout()
plt.show()

pd.DataFrame({
    "Quantity": ["Low-fidelity simulations", "High-fidelity experiments", "Best high-fidelity score", "Best gradient time (min)", "Best final %B"],
    "Value": [len(candidates), len(high_points), high_scores[best_idx], best_xy[0], best_xy[1]],
}).round(3)


## 8. Interactive optimization exercise

This dashboard lets students compare optimization strategies without editing Python code. It is designed for JupyterLite and uses only browser-side JavaScript after the data payload has been generated.

Use it to answer the questions below the plot. The **Show suggested answers** button provides model answers for classroom discussion.


In [ ]:
# Interactive optimization exercise.
# Run this cell once. Students do not need to edit the code below.

DASH_NX = 58
DASH_NY = 42


def _surface_payload(complexity):
    u = np.linspace(0, 1, DASH_NX)
    v = np.linspace(0, 1, DASH_NY)
    uu, vv = np.meshgrid(u, v)
    uv = np.stack([uu.ravel(), vv.ravel()], axis=1)
    z = normalized_score(uv, complexity=complexity).reshape(DASH_NY, DASH_NX)
    return {"nx": DASH_NX, "ny": DASH_NY, "z": [float(x) for x in z.ravel()]}


def _sample_records(points, scores):
    records = []
    for uv, score in zip(points, scores):
        xy = to_method_units(uv)
        records.append({"u": float(uv[0]), "v": float(uv[1]), "x": float(xy[0]), "y": float(xy[1]), "score": float(score)})
    return records


def _bo_result_for_dashboard(complexity, budget, exploration):
    """Fast BO-like surrogate search for the browser dashboard.

    The static section above uses a true Gaussian-process BO. For the dashboard we use
    a lightweight surrogate heuristic so the cell remains responsive in JupyterLite.
    """
    beta_map = {"focused": 0.18, "balanced": 0.32, "exploratory": 0.50}
    rng = np.random.default_rng(18)
    x_train = rng.uniform(0.05, 0.95, size=(4, 2))
    y_train = normalized_score(x_train, complexity=complexity)
    grid_axis = np.linspace(0.02, 0.98, 34)
    uu, vv = np.meshgrid(grid_axis, grid_axis)
    candidates = np.stack([uu.ravel(), vv.ravel()], axis=1)
    best_hist = [float(y_train.max())]

    for _ in range(max(1, budget - 4)):
        distances = np.sqrt(((candidates[:, None, :] - x_train[None, :, :]) ** 2).sum(axis=2))
        nearest = distances.min(axis=1)
        weights = np.exp(-0.5 * (distances / 0.18) ** 2)
        mu = (weights @ y_train) / (weights.sum(axis=1) + 1e-9)
        sigma = np.clip(nearest / 0.35, 0, 1)
        acq = mu + beta_map[exploration] * sigma
        acq[nearest < 0.035] = -np.inf
        next_x = candidates[int(np.argmax(acq))]
        next_y = normalized_score(next_x, complexity=complexity)
        x_train = np.vstack([x_train, next_x])
        y_train = np.append(y_train, next_y)
        best_hist.append(float(y_train.max()))

    best_idx = int(np.argmax(y_train))
    return {
        "samples": _sample_records(x_train, y_train),
        "path": _sample_records(x_train[[best_idx]], y_train[[best_idx]]),
        "best_score": float(y_train[best_idx]),
        "best_x": float(to_method_units(x_train[best_idx])[0]),
        "best_y": float(to_method_units(x_train[best_idx])[1]),
        "history": [float(v) for v in best_hist],
        "metric_label": "best score after BO-like search",
        "explanation": "Bayesian optimization uses previous experiments to update a surrogate model, then chooses the next experiment by balancing predicted score and uncertainty. This dashboard uses a lightweight surrogate approximation for browser speed.",
    }


def _gradient_result_for_dashboard(complexity, budget, start_name):
    starts = {"lower-left": (0.12, 0.18), "centre": (0.52, 0.52), "upper-right": (0.86, 0.82)}
    path, scores = gradient_ascent(complexity=complexity, start=starts[start_name], step_size=0.055, n_steps=budget)
    best_idx = int(np.argmax(scores))
    best_xy = to_method_units(path[best_idx])
    global_uv, global_score = best_grid_point(complexity=complexity)
    trapped = "likely local" if scores[best_idx] < global_score - 0.07 else "near-global"
    return {
        "samples": _sample_records(path, scores),
        "path": _sample_records(path, scores),
        "best_score": float(scores[best_idx]),
        "best_x": float(best_xy[0]),
        "best_y": float(best_xy[1]),
        "history": [float(v) for v in np.maximum.accumulate(scores)],
        "metric_label": f"{trapped} optimum",
        "explanation": "Gradient ascent follows local slope information. On rugged CRF surfaces the result can depend strongly on the starting point.",
    }


def _pso_result_for_dashboard(complexity, budget, exploration):
    params = {"focused": (0.55, 1.0, 1.55), "balanced": (0.66, 1.2, 1.35), "exploratory": (0.82, 1.45, 1.15)}[exploration]
    history, best_history, score_history = particle_swarm(
        complexity=complexity, n_particles=16, n_iter=budget, inertia=params[0], cognitive=params[1], social=params[2], random_state=22)
    final_points = history[-1]
    final_scores = normalized_score(final_points, complexity=complexity)
    best_uv = best_history[-1]
    best_xy = to_method_units(best_uv)
    return {
        "samples": _sample_records(final_points, final_scores),
        "path": _sample_records(best_history, score_history),
        "best_score": float(score_history[-1]),
        "best_x": float(best_xy[0]),
        "best_y": float(best_xy[1]),
        "history": [float(v) for v in score_history],
        "metric_label": "best swarm member",
        "explanation": "Particle swarm uses multiple particles. Each particle moves using its own best experience and the best experience of the swarm.",
    }


def _mf_result_for_dashboard(complexity, budget, exploration):
    sim_multiplier = {"focused": 7, "balanced": 11, "exploratory": 16}[exploration]
    n_exp = budget
    n_sim = budget * sim_multiplier
    candidates, low_scores, high_points, high_scores, best_idx = multi_fidelity_search(complexity=complexity, n_sim=n_sim, n_exp=n_exp, random_state=31)
    best_xy = to_method_units(high_points[best_idx])
    subset = np.argsort(low_scores)[-min(70, len(low_scores)):]
    samples = _sample_records(candidates[subset], low_scores[subset])
    high_records = _sample_records(high_points, high_scores)
    for item in high_records:
        item["high_fidelity"] = True
    return {
        "samples": samples + high_records,
        "path": _sample_records(high_points[[best_idx]], high_scores[[best_idx]]),
        "best_score": float(high_scores[best_idx]),
        "best_x": float(best_xy[0]),
        "best_y": float(best_xy[1]),
        "history": [float(v) for v in np.maximum.accumulate(high_scores)],
        "metric_label": f"{n_sim} simulations + {n_exp} experiments",
        "explanation": "Multi-fidelity optimization uses many cheap simulations to choose a smaller number of expensive high-fidelity experiments.",
    }

SURFACE_OPTIONS = ["smooth", "rugged"]
ALGORITHMS = ["Gradient ascent", "Particle swarm", "Bayesian optimization", "Multi-fidelity search"]
BUDGET_OPTIONS = [8, 16, 24]
EXPLORATION_OPTIONS = ["focused", "balanced", "exploratory"]
START_OPTIONS = ["lower-left", "centre", "upper-right"]

optimization_payload = {
    "surface_options": SURFACE_OPTIONS,
    "algorithms": ALGORITHMS,
    "budget_options": BUDGET_OPTIONS,
    "exploration_options": EXPLORATION_OPTIONS,
    "start_options": START_OPTIONS,
    "x_range": list(X_RANGE),
    "y_range": list(Y_RANGE),
    "surfaces": {},
    "results": {},
}

for complexity in SURFACE_OPTIONS:
    optimization_payload["surfaces"][complexity] = _surface_payload(complexity)
    global_uv, global_score = best_grid_point(complexity=complexity)
    optimization_payload["surfaces"][complexity]["global_best"] = {
        "u": float(global_uv[0]), "v": float(global_uv[1]),
        "x": float(to_method_units(global_uv)[0]), "y": float(to_method_units(global_uv)[1]),
        "score": float(global_score),
    }
    for budget in BUDGET_OPTIONS:
        for start in START_OPTIONS:
            key = f"{complexity}|Gradient ascent|budget={budget}|start={start}|exploration=balanced"
            optimization_payload["results"][key] = _gradient_result_for_dashboard(complexity, budget, start)
        for exploration in EXPLORATION_OPTIONS:
            key = f"{complexity}|Particle swarm|budget={budget}|start=centre|exploration={exploration}"
            optimization_payload["results"][key] = _pso_result_for_dashboard(complexity, budget, exploration)
            key = f"{complexity}|Bayesian optimization|budget={budget}|start=centre|exploration={exploration}"
            optimization_payload["results"][key] = _bo_result_for_dashboard(complexity, budget, exploration)
            key = f"{complexity}|Multi-fidelity search|budget={budget}|start=centre|exploration={exploration}"
            optimization_payload["results"][key] = _mf_result_for_dashboard(complexity, budget, exploration)

container_id = "ideas_opt_explorer_" + uuid.uuid4().hex[:8]
payload_json = json.dumps(optimization_payload)

html_template = '''
<div id="__CONTAINER_ID__" class="ideas-opt-dashboard">
  <style>
    #__CONTAINER_ID__ { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: #16495A; border: 1px solid #D9F7FF; border-radius: 16px; padding: 16px; background: #ffffff; max-width: 100%; width: 100%; box-sizing: border-box; overflow: visible; }
    #__CONTAINER_ID__ .header { display: flex; justify-content: space-between; gap: 12px; align-items: flex-start; margin-bottom: 12px; }
    #__CONTAINER_ID__ .brand { font-size: 0.88rem; color: #16495A; background: #D9F7FF; border-radius: 999px; padding: 5px 10px; white-space: nowrap; }
    #__CONTAINER_ID__ h3 { margin: 0 0 5px 0; }
    #__CONTAINER_ID__ p { margin: 4px 0; }
    #__CONTAINER_ID__ .controls { display: grid; grid-template-columns: repeat(5, minmax(130px, 1fr)); gap: 10px; margin: 12px 0 14px 0; }
    #__CONTAINER_ID__ label { font-size: 0.86rem; font-weight: 650; display: flex; flex-direction: column; gap: 4px; }
    #__CONTAINER_ID__ select { border: 1px solid #BDEFFF; border-radius: 8px; padding: 7px 8px; background: #fff; color: #16495A; font-size: 0.92rem; }
    #__CONTAINER_ID__ .layout { display: grid; grid-template-columns: minmax(460px, 1.55fr) minmax(280px, 0.9fr); gap: 14px; align-items: start; }
    #__CONTAINER_ID__ canvas { width: 100%; max-width: 100%; border: 1px solid #D9F7FF; border-radius: 12px; background: #ffffff; box-sizing: border-box; }
    #__CONTAINER_ID__ .card { border: 1px solid #D9F7FF; border-radius: 12px; padding: 10px; margin-bottom: 10px; background: #FAFEFF; }
    #__CONTAINER_ID__ .metric { font-size: 1.7rem; font-weight: 780; color: #DB3FC8; line-height: 1.1; }
    #__CONTAINER_ID__ .small { font-size: 0.88rem; color: #3d6170; }
    #__CONTAINER_ID__ .history { width: 100%; height: 76px; border: 1px solid #E5F8FD; border-radius: 8px; margin-top: 8px; }
    #__CONTAINER_ID__ textarea { width: 100%; min-height: 70px; border: 1px solid #BDEFFF; border-radius: 8px; padding: 8px; box-sizing: border-box; font-family: inherit; color: #16495A; }
    #__CONTAINER_ID__ .button-row { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 8px; }
    #__CONTAINER_ID__ button { border: 1px solid #43CEFF; border-radius: 999px; padding: 7px 12px; background: #ffffff; color: #16495A; cursor: pointer; font-weight: 650; }
    #__CONTAINER_ID__ button:hover { background: #D9F7FF; }
    #__CONTAINER_ID__ .answers { display: none; border-left: 4px solid #FFC000; padding: 8px 10px; background: #FFF2BF; border-radius: 8px; margin-top: 8px; font-size: 0.9rem; }
    @media (max-width: 900px) { #__CONTAINER_ID__ .controls { grid-template-columns: repeat(2, minmax(130px, 1fr)); } #__CONTAINER_ID__ .layout { grid-template-columns: 1fr; } }
  </style>
  <div class="header"><div><h3>Optimization — interactive exercise</h3><p class="small">Compare local, population-based, model-based, and multi-fidelity optimization on synthetic LC-MS method-development surfaces.</p></div><div class="brand">IDEAS — bos-ideas.com</div></div>
  <div class="controls">
    <label>Surface<select data-control="surface"><option value="smooth">Smooth response surface</option><option value="rugged" selected>Rugged CRF surface</option></select></label>
    <label>Optimizer<select data-control="algorithm"><option>Gradient ascent</option><option selected>Bayesian optimization</option><option>Particle swarm</option><option>Multi-fidelity search</option></select></label>
    <label>Budget<select data-control="budget"><option value="8">8 evaluations/iterations</option><option value="16" selected>16 evaluations/iterations</option><option value="24">24 evaluations/iterations</option></select></label>
    <label>Exploration<select data-control="exploration"><option value="focused">focused</option><option value="balanced" selected>balanced</option><option value="exploratory">exploratory</option></select></label>
    <label>Gradient start<select data-control="start"><option value="lower-left">lower-left</option><option value="centre" selected>centre</option><option value="upper-right">upper-right</option></select></label>
  </div>
  <div class="layout">
    <div><canvas width="780" height="560" data-role="main-canvas"></canvas><p class="small" data-role="caption"></p></div>
    <div>
      <div class="card"><div class="small">Best normalized CRF score</div><div class="metric" data-role="score">—</div><div class="small" data-role="best-method">—</div><canvas width="360" height="90" class="history" data-role="history-canvas"></canvas></div>
      <div class="card"><strong>Interpretation</strong><p class="small" data-role="interpretation"></p></div>
      <div class="card"><strong>Course questions</strong><p class="small">1. Why can this optimizer succeed or fail on this surface?</p><textarea data-answer="q1" placeholder="Write your answer here."></textarea><p class="small">2. Would you use this method for a real LC-MS method-development problem? Why?</p><textarea data-answer="q2" placeholder="Write your answer here."></textarea><div class="button-row"><button data-action="show-answers">Show suggested answers</button><button data-action="use-answers">Use suggested answers in boxes</button><button data-action="copy">Copy answers</button><button data-action="download">Download answers</button></div><div class="answers" data-role="suggested"></div></div>
    </div>
  </div>
</div>
<script type="application/json" id="__CONTAINER_ID___payload">__PAYLOAD_JSON__</script>
<script>
(function() {
  const root = document.getElementById("__CONTAINER_ID__");
  const payload = JSON.parse(document.getElementById("__CONTAINER_ID___payload").textContent);
  const canvas = root.querySelector('[data-role="main-canvas"]');
  const ctx = canvas.getContext('2d');
  const historyCanvas = root.querySelector('[data-role="history-canvas"]');
  const hctx = historyCanvas.getContext('2d');
  const controls = { surface: root.querySelector('[data-control="surface"]'), algorithm: root.querySelector('[data-control="algorithm"]'), budget: root.querySelector('[data-control="budget"]'), exploration: root.querySelector('[data-control="exploration"]'), start: root.querySelector('[data-control="start"]') };
  const colors = { cyan: '#43CEFF', magenta: '#DB3FC8', gold: '#FFC000', green: '#92D050', text: '#16495A', grid: '#E5F8FD' };
  function expandNotebookOutputArea() { ['.jp-OutputArea-output', '.jp-RenderedHTMLCommon', '.output_html', '.output_subarea'].forEach(sel => { const wrapper = root.closest(sel); if (wrapper && wrapper.style) { wrapper.style.maxHeight = 'none'; wrapper.style.overflow = 'visible'; wrapper.style.height = 'auto'; } }); }
  expandNotebookOutputArea();
  function resultKey() { const surface = controls.surface.value; const algorithm = controls.algorithm.value; const budget = controls.budget.value; const start = algorithm === 'Gradient ascent' ? controls.start.value : 'centre'; const exploration = algorithm === 'Gradient ascent' ? 'balanced' : controls.exploration.value; return `${surface}|${algorithm}|budget=${budget}|start=${start}|exploration=${exploration}`; }
  function currentResult() { return payload.results[resultKey()]; }
  function currentSurface() { return payload.surfaces[controls.surface.value]; }
  function scoreColor(value) { value = Math.max(0, Math.min(1, value)); if (value < 0.25) return '#FFFFFF'; if (value < 0.50) return '#D9F7FF'; if (value < 0.70) return '#43CEFF'; if (value < 0.87) return '#DB3FC8'; return '#FFC000'; }
  function sx(u) { const pad = 62; return pad + u * (canvas.width - 2 * pad); }
  function sy(v) { const pad = 62; return canvas.height - pad - v * (canvas.height - 2 * pad); }
  function drawSurface(surface) { const pad = 62; const w = canvas.width - 2 * pad; const h = canvas.height - 2 * pad; const cw = w / surface.nx; const ch = h / surface.ny; for (let j = 0; j < surface.ny; j++) { for (let i = 0; i < surface.nx; i++) { const z = surface.z[j * surface.nx + i]; ctx.fillStyle = scoreColor(z); ctx.fillRect(pad + i * cw, canvas.height - pad - (j + 1) * ch, cw + 1, ch + 1); } } }
  function drawAxes() { const pad = 62; ctx.strokeStyle = colors.grid; ctx.lineWidth = 1; for (let i = 0; i <= 5; i++) { const gx = pad + i * (canvas.width - 2 * pad) / 5; const gy = canvas.height - pad - i * (canvas.height - 2 * pad) / 5; ctx.beginPath(); ctx.moveTo(gx, pad); ctx.lineTo(gx, canvas.height - pad); ctx.stroke(); ctx.beginPath(); ctx.moveTo(pad, gy); ctx.lineTo(canvas.width - pad, gy); ctx.stroke(); } ctx.strokeStyle = colors.text; ctx.lineWidth = 1.2; ctx.strokeRect(pad, pad, canvas.width - 2 * pad, canvas.height - 2 * pad); ctx.fillStyle = colors.text; ctx.font = '13px sans-serif'; ctx.fillText('Gradient time (min)', canvas.width / 2 - 55, canvas.height - 18); ctx.save(); ctx.translate(20, canvas.height / 2 + 65); ctx.rotate(-Math.PI / 2); ctx.fillText('Final organic modifier (%B)', 0, 0); ctx.restore(); ctx.font = '11px sans-serif'; for (let i = 0; i <= 5; i++) { const u = i / 5; const xVal = payload.x_range[0] + u * (payload.x_range[1] - payload.x_range[0]); const yVal = payload.y_range[0] + u * (payload.y_range[1] - payload.y_range[0]); ctx.fillText(xVal.toFixed(0), sx(u) - 8, canvas.height - 42); ctx.fillText(yVal.toFixed(0), 32, sy(u) + 4); } }
  function drawMain() { const surface = currentSurface(); const result = currentResult(); ctx.clearRect(0, 0, canvas.width, canvas.height); drawSurface(surface); drawAxes(); const globalBest = surface.global_best; ctx.fillStyle = colors.gold; ctx.strokeStyle = colors.text; ctx.lineWidth = 1.5; ctx.beginPath(); const gx = sx(globalBest.u), gy = sy(globalBest.v); for (let k = 0; k < 5; k++) { const angle = -Math.PI / 2 + k * 2 * Math.PI / 5; const x = gx + Math.cos(angle) * 10; const y = gy + Math.sin(angle) * 10; if (k === 0) ctx.moveTo(x, y); else ctx.lineTo(x, y); } ctx.closePath(); ctx.fill(); ctx.stroke(); for (const p of result.samples || []) { ctx.beginPath(); ctx.fillStyle = p.high_fidelity ? colors.magenta : 'rgba(67,206,255,0.70)'; ctx.strokeStyle = 'white'; ctx.lineWidth = 0.8; ctx.arc(sx(p.u), sy(p.v), p.high_fidelity ? 5.2 : 3.6, 0, 2 * Math.PI); ctx.fill(); ctx.stroke(); } if (result.path && result.path.length > 0) { ctx.strokeStyle = colors.text; ctx.lineWidth = 2.0; ctx.beginPath(); result.path.forEach((p, idx) => { const x = sx(p.u), y = sy(p.v); if (idx === 0) ctx.moveTo(x, y); else ctx.lineTo(x, y); }); ctx.stroke(); result.path.forEach((p, idx) => { ctx.beginPath(); ctx.fillStyle = idx === result.path.length - 1 ? colors.green : colors.text; ctx.strokeStyle = 'white'; ctx.lineWidth = 0.8; ctx.arc(sx(p.u), sy(p.v), idx === result.path.length - 1 ? 6.5 : 3.6, 0, 2 * Math.PI); ctx.fill(); ctx.stroke(); }); } ctx.fillStyle = colors.text; ctx.font = '16px sans-serif'; ctx.fillText(`${controls.algorithm.value} on ${controls.surface.value} surface`, 64, 34); ctx.font = '12px sans-serif'; ctx.fillText('★ approximate global optimum; green endpoint = selected best', 64, 52); }
  function drawHistory() { const result = currentResult(); const hist = result.history || []; hctx.clearRect(0, 0, historyCanvas.width, historyCanvas.height); hctx.fillStyle = '#FFFFFF'; hctx.fillRect(0, 0, historyCanvas.width, historyCanvas.height); hctx.strokeStyle = colors.grid; hctx.strokeRect(0.5, 0.5, historyCanvas.width - 1, historyCanvas.height - 1); if (hist.length < 2) return; const minVal = Math.min(...hist, 0.0); const maxVal = Math.max(...hist, 1.0); const pad = 10; hctx.strokeStyle = colors.magenta; hctx.lineWidth = 2.2; hctx.beginPath(); hist.forEach((v, i) => { const x = pad + i * (historyCanvas.width - 2 * pad) / (hist.length - 1); const y = historyCanvas.height - pad - (v - minVal) / (maxVal - minVal + 1e-9) * (historyCanvas.height - 2 * pad); if (i === 0) hctx.moveTo(x, y); else hctx.lineTo(x, y); }); hctx.stroke(); }
  function suggestedAnswers() { const algorithm = controls.algorithm.value; const surface = controls.surface.value; const result = currentResult(); let q1 = ''; let q2 = ''; if (algorithm === 'Gradient ascent') { q1 = `Gradient ascent uses only local slope information. On the ${surface} surface it can move efficiently uphill, but on a rugged surface it may stop at a local optimum that depends on the starting point.`; q2 = 'For real LC-MS method development I would use it only when the response surface is expected to be smooth or when it is combined with many starting points. It is not ideal as the only global optimizer.'; } else if (algorithm === 'Particle swarm') { q1 = 'Particle swarm can succeed because many candidate methods explore the landscape simultaneously and share best-so-far information. It can still fail if the swarm collapses too early or the evaluation budget is too small.'; q2 = 'It is useful for rugged or multi-modal surfaces, especially as a global search strategy. The cost is that many evaluations may be needed.'; } else if (algorithm === 'Bayesian optimization') { q1 = 'Bayesian optimization learns a surrogate model from previous evaluations. It can succeed with few experiments by balancing predicted performance and uncertainty, but it can fail if the surrogate is misleading.'; q2 = 'It is well suited for expensive chromatographic experiments because it uses information from earlier experiments to choose the next method condition.'; } else { q1 = 'Multi-fidelity search can succeed when cheap simulations are correlated with the true experimental objective. It can fail when the simulator bias points the search away from the real optimum.'; q2 = 'It is attractive for LC-MS method development when many simulations are cheap but laboratory experiments are expensive. Experimental confirmation remains essential.'; } return {q1, q2, summary: `Current best score: ${result.best_score.toFixed(3)} at ${result.best_x.toFixed(1)} min and ${result.best_y.toFixed(1)} %B.`}; }
  function updateCards() { const result = currentResult(); root.querySelector('[data-role="score"]').textContent = result.best_score.toFixed(3); root.querySelector('[data-role="best-method"]').textContent = `${result.metric_label}: ${result.best_x.toFixed(1)} min, ${result.best_y.toFixed(1)} %B`; root.querySelector('[data-role="interpretation"]').textContent = result.explanation; root.querySelector('[data-role="caption"]').textContent = 'Background colour = normalized synthetic CRF score. Points/paths show the optimizer evaluations. The surface is a teaching example, not a validated LC-MS model.'; const ans = suggestedAnswers(); root.querySelector('[data-role="suggested"]').innerHTML = `<strong>Suggested answer 1:</strong> ${ans.q1}<br><br><strong>Suggested answer 2:</strong> ${ans.q2}<br><br><strong>Current result:</strong> ${ans.summary}`; }
  function updateAll() { const isGradient = controls.algorithm.value === 'Gradient ascent'; controls.start.disabled = !isGradient; controls.exploration.disabled = isGradient; drawMain(); drawHistory(); updateCards(); }
  Object.values(controls).forEach(control => control.addEventListener('change', updateAll));
  root.querySelector('[data-action="show-answers"]').addEventListener('click', () => { const box = root.querySelector('[data-role="suggested"]'); box.style.display = box.style.display === 'block' ? 'none' : 'block'; });
  root.querySelector('[data-action="use-answers"]').addEventListener('click', () => { const ans = suggestedAnswers(); root.querySelector('[data-answer="q1"]').value = ans.q1; root.querySelector('[data-answer="q2"]').value = ans.q2; root.querySelector('[data-role="suggested"]').style.display = 'block'; });
  function answerText() { const ans1 = root.querySelector('[data-answer="q1"]').value; const ans2 = root.querySelector('[data-answer="q2"]').value; return `HPLC 2026 — Optimization interactive exercise\nDr. Tijmen S. Bos / IDEAS / bos-ideas.com\n\nSurface: ${controls.surface.value}\nOptimizer: ${controls.algorithm.value}\nBudget: ${controls.budget.value}\nExploration: ${controls.exploration.value}\nStart: ${controls.start.value}\n\nAnswer 1:\n${ans1}\n\nAnswer 2:\n${ans2}\n`; }
  root.querySelector('[data-action="copy"]').addEventListener('click', async () => { try { await navigator.clipboard.writeText(answerText()); } catch (err) { console.log(err); } });
  root.querySelector('[data-action="download"]').addEventListener('click', () => { const blob = new Blob([answerText()], {type: 'text/plain'}); const url = URL.createObjectURL(blob); const a = document.createElement('a'); a.href = url; a.download = 'hplc2026_optimization_answers.txt'; document.body.appendChild(a); a.click(); document.body.removeChild(a); URL.revokeObjectURL(url); });
  updateAll();
})();
</script>
'''

html = html_template.replace("__CONTAINER_ID__", container_id).replace("__PAYLOAD_JSON__", payload_json)
display(HTML(html))


## 9. Take-home interpretation

Optimization in chromatographic method development is not just an algorithm choice; it is also a problem-definition exercise. The optimizer only sees the objective that we give it. If the CRF rewards the wrong behaviour, the optimizer may find a mathematically good but chemically poor method.

For the course discussion:

- **Gradient-based algorithms** are efficient when the response surface is smooth and differentiable, but they are vulnerable to local optima.
- **Particle swarm and related population methods** are useful on rugged landscapes, but may require many objective evaluations.
- **Bayesian optimization** is attractive when experiments are expensive because it uses a surrogate model and uncertainty to choose the next experiment.
- **Multi-fidelity optimization** is attractive when many cheap simulations can reduce the number of expensive laboratory experiments.
- **RL and BO can be complementary**: RL can exploit experience from interaction, while BO can explore uncertain response landscapes efficiently.
